# 2. Advection and resolution

## Why study advection?

Advection is the transport of a quantity by a moving flow. It is one of the simplest wave-propagation problems, which makes it ideal for learning how a finite-volume method represents moving features. A numerical solution can conserve the total mass and still distort the shape of a profile: sharp edges may become rounded, and small oscillations or phase errors may appear.

Here the density profile is a periodic top-hat: the central half of a 1 cm domain has density `1`, while the outside half has density `0.5`. The gas moves with velocity `u = 1 cm s^-1`. The run lasts 2 s, so the pattern travels twice around the periodic domain and returns to its starting position. This makes the final profile easy to compare with the initial condition.

The temperature is deliberately very small, so pressure forces are weak and the experiment is dominated by transport. The underlying RadHydropy calculation still uses the Euler solver and an ideal-gas equation of state; this notebook focuses on the advection behavior of the density field. Each run uses a temporary directory, so changing the resolution does not modify the example files.

## Governing equation and exact solution

For a constant velocity, the density transport reduces to the one-dimensional linear advection equation

$$\frac{\partial \rho}{\partial t} + u\frac{\partial \rho}{\partial x}=0.$$

with periodic boundary condition `rho(x + L, t) = rho(x, t)`. Its exact solution is a translation of the initial profile:

$$\rho(x,t)=\rho_0\left((x-ut)\bmod L\right).$$

In this example `L = 1 cm` and `u = 1 cm s^-1`, so the crossing time is `L/u = 1 s`. At `t = 1 s` and `t = 2 s`, the exact profile is again aligned with the initial profile. The dashed analytic curve constructed by the maintained example uses this translated top-hat solution.

## Finite-volume update and numerical diffusion

A finite-volume method stores the cell average `rho_i` and updates it from the flux entering and leaving the cell:

$$\rho_i^{n+1}=\rho_i^n-\frac{\Delta t}{\Delta x}\left(F_{i+1/2}^n-F_{i-1/2}^n\right).$$

For positive constant velocity, the exact physical flux is `F = u rho`; an upwind flux uses the state from the cell to the left of each face. The dimensionless Courant number

$$C=\frac{u\Delta t}{\Delta x}$$

measures how far the flow travels in one timestep relative to one cell. The YAML uses `CFL = 0.1`, keeping the update stable and limiting the amount of information transported per step. For a first-order piecewise-constant method (`order: 0` here), the leading truncation error behaves like diffusion, approximately

$$\frac{\partial \rho}{\partial t}+u\frac{\partial \rho}{\partial x} \approx D_{\rm num}\frac{\partial^2\rho}{\partial x^2}, \qquad D_{\rm num}\sim \frac{u\Delta x}{2}(1-C).$$

Thus a discontinuity spreads over several cells. Refining the mesh decreases `Delta x`, reduces the numerical diffusion, and makes the numerical curve closer to the exact translated profile.

Each run uses a temporary directory, so changing the resolution does not modify the example files.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.2.6', 'scipy==1.15.3', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
PINNED_MARKER = Path('/content/.radhydropy_dependencies_pinned_numpy_2_2_6')
if not PINNED_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
    PINNED_MARKER.touch()
    print('Dependencies installed. Restarting the Colab runtime once to load the pinned NumPy/SciPy binaries.')
    os.kill(os.getpid(), 9)
sys.path.insert(0, str(REPO))
%matplotlib inline
import matplotlib.pyplot as plt
import yaml
from example import example_utils
from example.Advection1D import advection1d
plt.rcParams['figure.figsize'] = (9, 5)

In [ ]:
# Make one isolated configuration for each resolution.
base_yaml = REPO / 'example' / 'Advection1D' / 'advection1d.yaml'
base = yaml.safe_load(base_yaml.read_text())
results = {}

for resolution in (50, 100, 200):
    run_dir = Path('/content') / f'radhydropy-advection-{resolution}'
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir()
    config = yaml.safe_load(yaml.safe_dump(base))
    config['par']['mesh']['grid_cells'] = resolution
    config['initial_condition']['grid_cells'] = resolution
    config['par']['output']['directory'] = str(run_dir)
    config['par']['simulation']['initial_condition_filename'] = 'InitialCondition.hdf5'
    config['example']['output_indices'] = [0, 5]
    config_file = run_dir / 'advection.yaml'
    config_file.write_text(yaml.safe_dump(config))
    advection1d.main(config_file)
    loaded = example_utils.load_nested_example_config(config_file)
    fields = example_utils.snapshot_physical_fields(run_dir / 'Output_005.hdf5', loaded)
    results[resolution] = fields

print('Completed resolutions:', sorted(results))

In [ ]:
from IPython.display import display
fig, axis = plt.subplots(figsize=(9, 5))
for resolution, fields in results.items():
    x = fields['radius_proper_code']
    rho = fields['rho_proper_code']
    axis.plot(x, rho, label=f'{resolution} cells')
axis.set_xlabel('position [code length]')
axis.set_ylabel('density [code units]')
axis.set_title('Numerical diffusion in periodic advection')
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

## How to read the plot

The exact profile has two sharp transitions: one where the top-hat rises and one where it falls. The numerical curves should preserve the overall mass and periodic shape, but the transitions are spread across a finite number of cells. At 50 cells, each cell represents `0.02 cm`; at 200 cells, each cell represents `0.005 cm`, so the same physical transition can be represented much more sharply.

Look for three effects:

- **Diffusion:** the density jump becomes a smooth ramp. This is the main first-order truncation error.
- **Resolution dependence:** the ramp occupies fewer physical centimetres as the mesh is refined.
- **Conservation:** the area under each density curve should remain approximately constant, even though the edge is smeared.

A useful quantitative follow-up is to compute an L1 error against the exact translated top-hat, `mean(abs(rho_numeric - rho_exact))`, for each resolution. For a discontinuous profile, the convergence rate is influenced by the edges and is not necessarily the formal smooth-solution order. Increasing resolution also increases the number of timesteps, so accuracy and computational cost must be balanced.

In [ ]:
# Optional: save all resolution runs to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/02_advection_and_resolution')
# save_dir.mkdir(parents=True, exist_ok=True)
# for resolution in results:
#     destination = save_dir / f'{resolution}_cells'
#     shutil.copytree(Path('/content') / f'radhydropy-advection-{resolution}', destination, dirs_exist_ok=True)
# print('Saved to:', save_dir)